# Inspect OpenGenome2 Data

Minimal notebook to inspect the local `jsonl.gz` data structure.

In [9]:
from pathlib import Path

data_root = Path('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json')
files = sorted(data_root.rglob('*.jsonl.gz'))
len(files), files[:5]

(1005,
 [PosixPath('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json/midtraining_specific/gtdb_v220_stitched/data_gtdb_test_chunk1.jsonl.gz'),
  PosixPath('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json/midtraining_specific/gtdb_v220_stitched/data_gtdb_train_chunk1.jsonl.gz'),
  PosixPath('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json/midtraining_specific/gtdb_v220_stitched/data_gtdb_train_chunk10.jsonl.gz'),
  PosixPath('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json/midtraining_specific/gtdb_v220_stitched/data_gtdb_train_chunk11.jsonl.gz'),
  PosixPath('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json/midtraining_specific/gtdb_v220_stitched/data_gtdb_train_chunk12.jsonl.gz')])

In [10]:
from collections import Counter

Counter(str(p.relative_to(data_root).parent) for p in files) # 每个数据来源目录下面有多少个压缩的JSONL文件

Counter({'midtraining_specific/ncbi_eukaryotic_genomes/batch1': 95,
         'midtraining_specific/ncbi_eukaryotic_genomes/batch2': 94,
         'midtraining_specific/ncbi_eukaryotic_genomes/batch3': 94,
         'midtraining_specific/ncbi_eukaryotic_genomes/batch4': 94,
         'midtraining_specific/ncbi_eukaryotic_genomes/batch5': 94,
         'midtraining_specific/ncbi_eukaryotic_genomes/batch6': 94,
         'midtraining_specific/ncbi_eukaryotic_genomes/batch7': 94,
         'midtraining_specific/ncbi_eukaryotic_genomes/batch8': 94,
         'pretraining_or_both_phases/metagenomes': 82,
         'pretraining_or_both_phases/eukaryotic_genic_windows': 42,
         'midtraining_specific/gtdb_v220_stitched': 36,
         'pretraining_or_both_phases/gtdb_v220_imgpr': 36,
         'pretraining_or_both_phases/mrna_splice_promoter': 21,
         'pretraining_or_both_phases/mrna': 13,
         'pretraining_or_both_phases/imgvr_untagged': 6,
         'pretraining_or_both_phases/ncrna': 6,
 

| 目录名 | 大概含义 | 直觉理解 |
|---|---|---|
| `gtdb_v220_stitched` | GTDB v220 细菌 / 古菌基因组数据，经过 stitched 处理 | Prokaryotic whole-genome / long-context 数据，用来学习细菌、古菌基因组的长程结构 |
| `gtdb_v220_imgpr` | GTDB v220 + IMG/PR | 细菌 / 古菌基因组 + 质粒相关序列，覆盖原核生物和可移动遗传元件 |
| `ncbi_eukaryotic_genomes/batch1-8` | NCBI 真核生物基因组，分成多个 batch | 动物、植物、真菌、原生生物等真核 whole-genome 数据 |
| `metagenomes` | 宏基因组 contigs | 环境样本里拼出来的混合微生物序列，物种来源复杂 |
| `eukaryotic_genic_windows` | 真核基因相关窗口 | 以 gene / coding region 附近为中心切出来的片段，用来强化真核基因结构 |
| `mrna` | mRNA 序列 | 成熟转录本序列，偏 exon / transcript 信息 |
| `mrna_splice_promoter` | mRNA + splice / promoter 相关窗口 | 强化剪接位点、启动子等功能区域 |
| `imgvr_untagged` | IMG/VR 病毒序列，未加 tag 版本 | 多数是病毒 / 噬菌体序列 |
| `ncrna` | non-coding RNA | 不翻译成蛋白的 RNA 序列，比如 rRNA、tRNA、lncRNA 等 |
| `imgpr` | IMG/PR | 质粒序列，也可能包含 phage-plasmid / mobile genetic elements |
| `organelle` | 细胞器基因组 | 线粒体、叶绿体等小型基因组 |
| `promoters` | 启动子区域 | 基因转录起始附近的 regulatory sequence |

In [11]:
import gzip, json

sample_file = data_root / 'pretraining_or_both_phases/mrna_splice_promoter/data_mrna_stitch_train_chunk1.jsonl.gz'
with gzip.open(sample_file, 'rt', encoding='utf-8') as f:
    rows = [json.loads(next(f)) for _ in range(3)]

print(f"{len(rows)= }")
print(f"{rows[0].keys() = }")
print(f"{rows[0] = }")
print(f"{[len(x['text']) for x in rows] = }")


len(rows)= 3
rows[0].keys() = dict_keys(['text'])
rows[0] = {'text': 'TTTTCTATAAAAAGTGAAATTATATTTTGCCCTTGTGAAATTGTGTTGATTGGTCCTGTAGTTTATGCACTGTAAAATGATATTGAACTTGCTTATGCATCTACTCTTTGACTGTGATTTGATTTGAAACTACACACTTGTAATGTAGGAACTCTATTCCTGAATTATTTTGCCACTTGAGTAGCTAGCTTATGACTTATGTTTTGTTCAGCAGTGTACAAGTATGCTAATGTGTCGATTTATTTGGAGAGGGGTGAAAAAAATAAATCTGATCCTATCATCAACTGATGCTTGCTGGATTTTATGCATGGAGTAGAAATCTTTTTGAATACTTTGTATATTTAAGGTGTTGCAAACCTATGACTGAGAGATGCCCATCTTCTGGGTGAAAGTAATGCAATTCAGACAATGTATAGTACTGAATAATTTTTATTCTCTTGTGATACCTCTTGGGTGGCCGTTTCTTCGGTTCCATTCTTAAGAACTACATGTAGGGCGCAGACAATGCCTCTCATTCACAGCACCATGTGGCTGTGAACATCACTGATAGCATGCTACCCTTTTCAATGACTCAGCAATTCTCACGCTGACTGAACAATGCCTTTCTGGCCTACTCTTAATTCTACCGCTGTCCGCCAGCGGGTCGTTATGTACGGTACTTATCCCGTAAAAGCAAACACCTGCGATGCTTTCGGGCTCTAGCTAGAAATTCTATTGTCCATCTCAGGATCAAAACTTCGAATCCTTCTACCCGAAGAACGGGATCTTGGTCCAGTCTGGTTGAAGCGTTGGTCGGGCGTTTGGCGGGTAGACTCGCGGGTCGGGCGTCCGTTTGTGGGAGGCGGCGGCGTCGCCGCCGGTCTGTTTGACCTGCCGCTCCTCGATGGACCGCTGCTTGTCCGTCTGGTCCCCGAATGGCGTAAGGAAGAA

In [12]:
import statistics

with gzip.open(sample_file, 'rt', encoding='utf-8') as f:
    lengths = [len(json.loads(line)['text']) for _, line in zip(range(100), f)]

{
    'n': len(lengths),
    'min': min(lengths),
    'max': max(lengths),
    'mean': statistics.mean(lengths),
}

{'n': 100, 'min': 1128, 'max': 9667, 'mean': 3025.19}

In [13]:
import gzip
import json
from pathlib import Path
from collections import defaultdict, Counter
from tqdm import tqdm

data_root = Path('/inspire/hdd/project/reasoning/public/activations/evo2_7b/opengenome2/json')
files = sorted(data_root.rglob('*.jsonl.gz'))

schema_by_group = defaultdict(Counter)

# 每个 group 的字符统计
char_by_group = defaultdict(Counter)

# 记录出现了非 A/C/G/T/N 字符的样本
bad_char_examples = []

# 记录出现 N 的样本
n_examples = []

allowed_with_n = set("ACGTN")
allowed_actg = set("ACGT")

for p in tqdm(files):
    group = str(p.relative_to(data_root).parent)

    with gzip.open(p, "rt") as f:
        for i, line in enumerate(f):
            obj = json.loads(line)

            # 1. 检查 schema
            schema = tuple(sorted(obj.keys()))
            schema_by_group[group].update([schema])

            # 2. 检查 text 字段中的字符
            seq = obj["text"].upper()

            chars = Counter(seq)
            char_by_group[group].update(chars)

            # 是否存在非 A/C/G/T/N 字符
            bad_chars = set(seq) - allowed_with_n
            if bad_chars and len(bad_char_examples) < 20:
                bad_char_examples.append({
                    "file": str(p),
                    "line": i,
                    "bad_chars": bad_chars,
                    "preview": seq[:200],
                })

            # 是否存在 N
            if "N" in chars and len(n_examples) < 20:
                n_examples.append({
                    "file": str(p),
                    "line": i,
                    "num_N": chars["N"],
                    "length": len(seq),
                    "preview": seq[:200],
                })

            # 每个文件只看前 10 行
            if i >= 9:
                break

print("\n" + "=" * 100)
print("Schema by group")
for group, counter in schema_by_group.items():
    print("=" * 100)
    print(group)
    print(counter)

print("\n" + "=" * 100)
print("Character stats by group")
for group, counter in char_by_group.items():
    total = sum(counter.values())
    n_count = counter.get("N", 0)

    print("=" * 100)
    print(group)
    print("chars:", counter)
    print("total chars:", total)
    print("N count:", n_count)
    print("N ratio:", n_count / total if total > 0 else 0)

print("\n" + "=" * 100)
print("Bad char examples, excluding A/C/G/T/N")
print(bad_char_examples)

print("\n" + "=" * 100)
print("N examples")
print(n_examples)

100%|██████████| 1005/1005 [2:08:04<00:00,  7.65s/it] 


Schema by group
midtraining_specific/gtdb_v220_stitched
Counter({('text',): 360})
midtraining_specific/imgpr
Counter({('record', 'text'): 40})
midtraining_specific/ncbi_eukaryotic_genomes/batch1
Counter({('text',): 870})
midtraining_specific/ncbi_eukaryotic_genomes/batch2
Counter({('text',): 836})
midtraining_specific/ncbi_eukaryotic_genomes/batch3
Counter({('text',): 844})
midtraining_specific/ncbi_eukaryotic_genomes/batch4
Counter({('text',): 876})
midtraining_specific/ncbi_eukaryotic_genomes/batch5
Counter({('text',): 868})
midtraining_specific/ncbi_eukaryotic_genomes/batch6
Counter({('text',): 857})
midtraining_specific/ncbi_eukaryotic_genomes/batch7
Counter({('text',): 868})
midtraining_specific/ncbi_eukaryotic_genomes/batch8
Counter({('text',): 868})
pretraining_or_both_phases/eukaryotic_genic_windows
Counter({('text',): 420})
pretraining_or_both_phases/gtdb_v220_imgpr
Counter({('record', 'text'): 360})
pretraining_or_both_phases/imgvr_untagged
Counter({('text',): 60})
pretraini